# 04 — HRV Results Summary and Publication Outputs
This notebook consolidates outputs from all prior notebooks (00–03), computes global metrics, and prepares final figures and tables for publication.

In [ ]:
# =====================================================
# 0. Universal Project Path Setup (same as 00–03)
# =====================================================
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Resolve project root ---
project_root = Path.cwd().resolve()
if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

# --- Define directories ---
data_processed = project_root / "data" / "processed"
results_dir = project_root / "results"
manuscript_dir = project_root / "manuscript"
figures_dir = results_dir / "figures"
tables_dir = manuscript_dir / "tables"

for d in [results_dir, manuscript_dir, figures_dir, tables_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("📁 Project root:", project_root)
print("📂 Results directory:", results_dir)
print("📂 Manuscript tables:", tables_dir)
print("📂 Figures:", figures_dir)

In [ ]:
# =====================================================
# 1. Load prior results (baseline, calibrated, sequence models)
# =====================================================
baseline_path = results_dir / "baseline_results_all.xlsx"
calibrated_path = results_dir / "Calibrated_HRV_Predictions_All.csv"

baseline = pd.read_excel(baseline_path) if baseline_path.exists() else None
calibrated = pd.read_csv(calibrated_path) if calibrated_path.exists() else None

print("✅ Loaded available datasets:")
if baseline is not None:
    print(f"Baseline results → {baseline.shape}")
if calibrated is not None:
    print(f"Calibrated predictions → {calibrated.shape}")

# Load training performance summaries if available
from pathlib import Path
lstm_fig = results_dir / "lstm_training_curve.png"
cnn_fig = results_dir / "cnn_training_curve.png"
print("LSTM Curve Exists:", lstm_fig.exists(), "CNN Curve Exists:", cnn_fig.exists())

In [ ]:
# =====================================================
# 2. Compute Global Summary Metrics
# =====================================================
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

metrics_summary = []
if calibrated is not None:
    for ds in calibrated['dataset'].unique():
        subset = calibrated[calibrated['dataset'] == ds]
        y_true = subset['y_true']
        y_pred = np.round(subset['y_pred_cal'])
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        auc = roc_auc_score(y_true, subset['y_pred_cal']) if len(np.unique(y_true)) == 2 else np.nan
        metrics_summary.append({
            'Dataset': ds,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-score': f1,
            'ROC AUC': auc
        })
summary_df = pd.DataFrame(metrics_summary)
display(summary_df)

In [ ]:
# =====================================================
# 3. Figure 8 – Cross-Dataset Comparison Bar Chart
# =====================================================
if not summary_df.empty:
    plt.figure(figsize=(8,6))
    melted = summary_df.melt(id_vars='Dataset', var_name='Metric', value_name='Score')
    sns.barplot(data=melted, x='Dataset', y='Score', hue='Metric', palette='tab10')
    plt.title('Figure 8. Cross-Dataset Performance Comparison (Calibrated)')
    plt.ylabel('Score')
    plt.xlabel('Dataset')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    out_file = figures_dir / 'Figure8_CrossDatasetComparison.png'
    plt.savefig(out_file, dpi=300)
    print(f'✅ Figure 8 saved to {out_file}')
    plt.show()

In [ ]:
# =====================================================
# 4. Figure 9 – Correlation Heatmap (Calibrated vs Uncalibrated)
# =====================================================
if calibrated is not None:
    plt.figure(figsize=(6,5))
    corr = calibrated[['y_pred_cal','y_pred_uncal']].corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Figure 9. Correlation: Calibrated vs Uncalibrated Predictions')
    out_file = figures_dir / 'Figure9_CalibrationCorrelation.png'
    plt.tight_layout()
    plt.savefig(out_file, dpi=300)
    print(f'✅ Figure 9 saved to {out_file}')
    plt.show()

In [ ]:
# =====================================================
# 5. Table 5 – Final Summary Metrics (for Publication)
# =====================================================
if not summary_df.empty:
    table5_path = tables_dir / 'Table5_SummaryMetrics.csv'
    summary_df.to_csv(table5_path, index=False)
    print(f'✅ Table 5 saved → {table5_path}')
    display(summary_df)

### ✅ Notebook Outputs:
- **Figures:**
  - Figure 8 – Cross-dataset performance comparison
  - Figure 9 – Correlation heatmap (calibrated vs uncalibrated)
- **Tables:**
  - Table 5 – Summary metrics (Accuracy, Precision, Recall, F1, ROC AUC)

These are automatically exported to:
- `/results/figures/`
- `/manuscript/tables/`